# EXP-0005 — comparación exploratoria Heat + Burgers

Tres arquitecturas con parámetros cercanos (3.648–3.655), tres semillas y los mismos pools Sobol scrambled fijos. Heat es conducción 1D en una varilla con referencia analítica; Burgers es el benchmark viscoso transitorio de Raissi/PINNacle. Las tablas son mediciones por semilla; los puntos diamante muestran media ± desviación estándar.

Esta evaluación valida ejecución y artefactos. No se preregistraron umbrales numéricos, por lo que no cuenta como aceptación científica ni como claim SOTA. Campos con signo permanecen lineales; pérdidas, magnitudes de gradiente/residual y errores positivos se muestran en escala logarítmica. El error absoluto usa `log10(|error|)`; los ceros exactos no reciben epsilon.

Datos completos: [`quantified-signals.csv`](../../var/agent_runs/heat-burgers-architecture-screen/analysis/quantified-signals.csv), [`signal-summary.csv`](../../var/agent_runs/heat-burgers-architecture-screen/analysis/signal-summary.csv), [`results.html`](../../var/agent_runs/heat-burgers-architecture-screen/analysis/architecture-screen-comparison.html).

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd()
if not (ROOT / 'var/agent_runs/heat-burgers-architecture-screen/analysis').exists():
    ROOT = ROOT.parents[1]
ANALYSIS = ROOT / 'var/agent_runs/heat-burgers-architecture-screen/analysis'
seed_results = pd.read_csv(ANALYSIS / 'seed-results.csv')
primary_summary = pd.read_csv(ANALYSIS / 'primary-summary.csv')
signal_summary = pd.read_csv(ANALYSIS / 'signal-summary.csv')


## Métricas principales

Errores positivos graficados en eje log. `n=3` semillas por arquitectura.

In [ ]:
primary_summary[primary_summary['signal'].isin([
'heat/field_relative_l2',
'burgers/field_relative_l2',
'burgers/front_field_rmse_normalized'])][['architecture', 'description', 'n', 'mean', 'std', 'min', 'max', 'unit']]

In [ ]:
colors = {'rbf_fixed':'#3366cc', 'mlp_tanh':'#dc3912', 'bspline_cubic':'#109618'}
metrics = [
    ('heat/field_relative_l2', 'Heat — error relativo L2'),
    ('burgers/field_relative_l2', 'Burgers — error relativo L2'),
    ('burgers/front_field_rmse_normalized', 'Burgers — RMSE del frente normalizado'),
]
fig = make_subplots(rows=1, cols=3, subplot_titles=[label for _, label in metrics])
for col, (metric, _) in enumerate(metrics, start=1):
    for model_id, group in seed_results.groupby('model_id', sort=False):
        for row in group.to_dict('records'):
            fig.add_trace(go.Scatter(x=[row['architecture']], y=[row[metric]],
                mode='markers', name=f"{row['architecture']} · semilla {int(row['seed'])}",
                legendgroup=model_id, marker={'color':colors[model_id], 'size':10}),
                row=1, col=col)
        values = group[metric]
        fig.add_trace(go.Scatter(x=[group.iloc[0]['architecture']], y=[values.mean()],
            mode='markers', showlegend=False, marker={'color':colors[model_id], 'size':15, 'symbol':'diamond'},
            error_y={'type':'data', 'array':[values.std(ddof=1)], 'visible':True}), row=1, col=col)
    fig.update_yaxes(type='log', title_text='Error (escala log)', row=1, col=col)
    fig.update_xaxes(tickangle=-20, row=1, col=col)
fig.update_layout(template='plotly_white', height=570, width=1550,
    title='EXP-0005 · errores de validación independiente', hovermode='closest')
fig


## Señales medidas de puntos, residual y recursos

La tabla completa conserva una fila por señal, problema, arquitectura y semilla; aquí se muestra el agregado descriptivo.

In [ ]:
signal_summary[signal_summary['signal'].str.contains('sampling|residual|resource_seed_campaign', regex=True)][['architecture','problem','signal','n','mean','std','min','max','unit']]

## Lectura limitada

El screening favorece B-spline para el error de Heat y RBF para los dos errores de Burgers seleccionados. Es una señal de especialización por problema, no una conclusión general: cada estimación usa tres semillas y no se realizó ajuste de hiperparámetros ni comparación con métodos SOTA externos. La utilización y memoria GPU se informan como señales agregadas de campaña (Heat + Burgers), no por PDE; la campaña usó slots GPU 0/1 asignados por semilla.